In [2]:
!pip install --no-index --find-links=/kaggle/input/torch-geometric /kaggle/input/torch-geometric/tg/torch_geometric-2.7.0-py3-none-any.whl
!rm -rf /kaggle/working/*

Looking in links: /kaggle/input/torch-geometric
Processing /kaggle/input/torch-geometric/tg/torch_geometric-2.7.0-py3-none-any.whl
torch-geometric is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.


In [ ]:
import os
import torch
from torch_geometric.data import Data
import torch_geometric.transforms as TG
from tqdm import tqdm

download_path = "/kaggle/input/shapenet"

def custom_read_obj(file_path):
    pos = []
    faces = []
    
    with open(file_path, 'r') as f:
        for line in f:
            components = line.strip().split() # split() without args handles multiple spaces and trailing spaces
            if not components:
                continue
            if components[0] == 'v':
                pos.append([float(x) for x in components[1:4]])
            elif components[0] == 'f':
                # OBJ indices are 1-based, convert to 0-based. Only take the vertex index.
                # Handle f v1 v2 v3 ... and f v1/vt1/vn1 ...
                indices = [int(c.split('/')[0]) - 1 for c in components[1:]]
                # Triangulate polygons (simple fan triangulation)
                for i in range(1, len(indices) - 1):
                    faces.append([indices[0], indices[i], indices[i+1]])

    if not pos:
        return Data(pos=torch.empty((0, 3)), face=torch.empty((2, 0)))

    pos_tensor = torch.tensor(pos, dtype=torch.float)
    
    if faces:
        face_tensor = torch.tensor(faces, dtype=torch.long).t().contiguous()
    else:
        face_tensor = torch.empty((3, 0), dtype=torch.long)
        
    return Data(pos=pos_tensor, face=face_tensor)

def visualize_point_normal_data(data_tensor):
    import matplotlib.pyplot as plt
    from mpl_toolkits.mplot3d import Axes3D
    
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    
    x = data_tensor[:, 0].numpy()
    y = data_tensor[:, 1].numpy()
    z = data_tensor[:, 2].numpy()
    
    ax.scatter(x, y, z, c='b', marker='o')
    
    ax.set_xlabel('X Label')
    ax.set_ylabel('Y Label')
    ax.set_zlabel('Z Label')
    
    plt.show()

def visualize_pc(pos, title="Point Cloud", density=2.5):
    """
    Visualizes a point cloud using matplotlib.
    pos: (N, 3) tensor
    """
    import matplotlib.pyplot as plt

    fig = plt.figure(figsize=(5, 5))
    ax = fig.add_subplot(111, projection='3d')
    if torch.is_tensor(pos):
        pos = pos.cpu().detach().numpy()
        
    ax.scatter(pos[:, 0], pos[:, 1], pos[:, 2], s=density)
    ax.set_title(title)
    plt.show()


def process_shapenet_mesh(data_folder, output_folder=None, num_points=1024, max_process_count=float('inf')):
    """
    Pulls mesh data from data_folder, converts to point normal form (x, y, z, nx, ny, nz),
    and stores them.
    
    Args:
        data_folder (str): root directory of the dataset (e.g. download_path)
        output_folder (str, optional): directory to save the processed .pt files
        num_points (int): number of points to sample

    Rets:
        list of Data objects
    """
    # TG.SamplePoints with include_normals=True samples points and computes face normals
    sampler = TG.SamplePoints(num=num_points, include_normals=True)
    
    processed_count = 0
    errors = []

    ret = []
    
    print(f"Scanning {data_folder} for models...")
    
    # We walk through the directory structure
    for root, dirs, files in os.walk(data_folder):
        for file in files:
            if processed_count >= max_process_count:
                break
            if file == "model_normalized.obj":
                file_path = os.path.join(root, file)
                
                # Extract category and model_id from path
                # Expected structure: .../category_id/model_id/models/model_normalized.obj
                path_parts = os.path.normpath(file_path).split(os.sep)
                
                try:
                    if 'models' in path_parts:
                        models_index = path_parts.index('models')
                        model_id = path_parts[models_index - 1]
                        category_id = path_parts[models_index - 2]
                    else:
                        continue
                        
                    # Read the mesh object using custom reader
                    data = custom_read_obj(file_path)
                    
                    if data.pos.size(0) == 0:
                        print(f"Skipping empty mesh: {file_path}")
                        continue

                    # Transformation to sample points and normals
                    data = sampler(data)
                    
                    # NOTE: Uncomment to visualize
                    # visualize_pc(data.pos, density = 2.5)

                    point_normal_data = torch.cat([data.pos, data.normal], dim=1)
                    
                    # Store data in pt file for later use
                    if output_folder:
                        save_cat_dir = os.path.join(output_folder, category_id)
                        os.makedirs(save_cat_dir, exist_ok=True)
                        output_path = os.path.join(save_cat_dir, f"{model_id}.pt")
                        torch.save(point_normal_data, output_path)

                    ret.append(data)
                    
                    processed_count += 1
                    if processed_count % 100 == 0:
                        print(f"Processed {processed_count} meshes...")
                        
                except Exception as e:
                    print(f"Error processing {file_path}: {e}")
                    errors.append((file_path, str(e)))
                    
    print(f"Finished. Total processed: {processed_count}")
    if errors:
        print(f"Encountered {len(errors)} errors.")
    return ret

# Example Usage:
# output_dir = "./processed_data"

if __name__ == "__main__":
    process_shapenet_mesh(download_path, output_folder="output_dir", num_points=2048)
    # try fetching the pt datas stored in output_dir
    datas = []
    data_dir = "output_dir"
    # loop over elements in data_dir
    for category in os.listdir(data_dir):
        category_path = os.path.join(data_dir, category)
        if os.path.isdir(category_path):
            for file in os.listdir(category_path):
                if file.endswith(".pt"):
                    file_path = os.path.join(category_path, file)
                    data_tensor = torch.load(file_path)
                    datas.append(data_tensor)
    print(f"Loaded {len(datas)} processed point-normal data files.")
    print(datas)